# Estrazione embedding ChemBERTa

Questo notebook estrae e salva **solo gli embedding grezzi** di ChemBERTa.
La riduzione dimensionale (PCA) viene eseguita successivamente, all'interno della pipeline di training/CV, per evitare leakage.

Vengono prodotte due rappresentazioni:
- `raw`: SMILES originale;
- `clean`: SMILES con `[*]` rimosso.

In [ ]:
from pathlib import Path
import os

import torch
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from pymongo import MongoClient
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel


# Impostazioni principali
COLLECTION_NAME = "biceranoPolymers"
TARGET = "Tg"
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
BATCH_SIZE = 32

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()
while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent
if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

OUTPUT_DIR = ROOT / "data" / "embeddings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR.resolve())

## Caricamento del dataset

In [ ]:
load_dotenv(ROOT / ".env")
MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    raise RuntimeError("MONGO_URI non trovato nel file .env")

client = MongoClient(MONGO_URI)
db = client["PolymerPrediction"]
collection = db[COLLECTION_NAME]

# Ordinamento esplicito per rendere riproducibile l'ordine dei campioni.
docs = list(collection.find({}).sort("_id", 1))
df = pd.DataFrame(docs)

if df.empty:
    raise RuntimeError(f"La collection {COLLECTION_NAME!r} e vuota")

# Espande il dizionario properties mantenendo le proprieta come colonne normali.
prop_df = pd.json_normalize(df["properties"])
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

# Mantiene soltanto campioni effettivamente utilizzabili per questo target.
df = df.dropna(subset=["smiles", TARGET]).copy()
df = df[~df["smiles"].astype(str).str.contains(r"\[\[", na=False)].copy()

# L'_id e la chiave usata per riallineare embedding e target nel notebook di training.
df["_id"] = df["_id"].astype(str)
df = df.reset_index(drop=True)

assert df["_id"].is_unique, "Sono presenti _id duplicati"

print("Collection:", COLLECTION_NAME)
print("Campioni utilizzabili:", len(df))
print("Target:", TARGET)
df[["_id", "smiles", TARGET]].head()

## Caricamento di ChemBERTa

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
encoder.eval()

print("Hidden size:", encoder.config.hidden_size)

## Estrazione degli embedding

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    """Media dei token reali, ignorando il padding introdotto dal batching."""
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def chemberta_embeddings(smiles_list, batch_size=BATCH_SIZE):
    """Estrae un embedding fisso per ciascuno SMILES."""
    chunks = []

    for start in tqdm(range(0, len(smiles_list), batch_size), desc="ChemBERTa"):
        batch = smiles_list[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = encoder(**inputs)
            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])

        chunks.append(pooled.cpu().numpy())

    return np.vstack(chunks)

In [ ]:
raw_smiles = df["smiles"].astype(str).tolist()
clean_smiles = [s.replace("[*]", "") for s in raw_smiles]

raw_array = chemberta_embeddings(raw_smiles)
clean_array = chemberta_embeddings(clean_smiles)

print("Raw shape:", raw_array.shape)
print("Clean shape:", clean_array.shape)

assert raw_array.shape[0] == len(df)
assert clean_array.shape == raw_array.shape

## Salvataggio

In [ ]:
def save_embeddings(array, variant):
    emb = pd.DataFrame(array, columns=[f"emb_{i}" for i in range(array.shape[1])])
    out = pd.concat([df[["_id"]].copy(), emb], axis=1)

    filename = f"chemberta_{COLLECTION_NAME}_{variant}.csv"
    path = os.path.join(OUTPUT_DIR, filename)
    out.to_csv(path, index=False)

    print(f"Salvato: {path}  shape={out.shape}")
    return out


raw_df = save_embeddings(raw_array, "raw")
clean_df = save_embeddings(clean_array, "clean")